In [ ]:
!pip install -q transformers beautifulsoup4 requests nltk wordcloud matplotlib

In [ ]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
def get_imdb_reviews(movie_id, max_reviews=10):
    url = f"https://www.imdb.com/title/{movie_id}/reviews"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    reviews = soup.find_all("div", class_="text show-more__control")
    review_texts = [rev.get_text() for rev in reviews[:max_reviews]]
    return review_texts

def summarize_reviews(reviews):
    summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
    all_reviews = " ".join(reviews)
    if len(all_reviews) > 1024:
        all_reviews = all_reviews[:1024]
    summary = summarizer(all_reviews, max_length=100, min_length=30, do_sample=False)
    return summary[0]['summary_text']

def analyze_sentiment(reviews):
    sia = SentimentIntensityAnalyzer()
    sentiments = []
    for review in reviews:
        score = sia.polarity_scores(review)
        sentiments.append(score)
    return sentiments

def create_wordcloud(text):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud of Reviews")
    plt.show()


In [ ]:
from IPython.display import display
import ipywidgets as widgets

def interactive_input():
    movie_input = widgets.Text(
        value='tt4154796',  # Default: Interstellar
        placeholder='Enter IMDb ID (e.g., tt1375666)',
        description='IMDb ID:',
        disabled=False
    )
    display(movie_input)

    button = widgets.Button(description="Run Analysis")
    display(button)

    def on_button_click(b):
        movie_id = movie_input.value
        run_pipeline(movie_id)

    button.on_click(on_button_click)

interactive_input()

Text(value='tt4154796', description='IMDb ID:', placeholder='Enter IMDb ID (e.g., tt1375666)')

Button(description='Run Analysis', style=ButtonStyle())